In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import pickle
import os
import random
from collections import defaultdict


In [2]:
RANDOM_SEED = 42
TEST_SIZE = 0.4
VAL_TEST_RATIO = 0.5
OUTPUT_DIR = 'splits_v2'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

data = pd.read_csv('../CR_real_masks_more_labeled_veritices_agreed.csv')
data['node_id1'] -= 1
data['node_id2'] -= 1

In [3]:
labels = data['label_id1'].unique().tolist() + data['label_id2'].unique().tolist()
labels = sorted(list(set([l for l in labels if l != 'masked'])))
print(f"Labels: {labels}")

df = pd.concat([
    data[['node_id1', 'label_id1']].rename(columns={'node_id1': 'node_id', 'label_id1': 'label'}),
    data[['node_id2', 'label_id2']].rename(columns={'node_id2': 'node_id', 'label_id2': 'label'})
], ignore_index=True)
df = df.drop_duplicates(subset=['node_id'])

max_node = max(data['node_id1'].max(), data['node_id2'].max())

known_nodes = []
unknown_nodes = []

for _, row in tqdm(df.iterrows(), desc="Processing nodes"):
    if row['label'] == 'masked':
        unknown_nodes.append(row['node_id'])
    else:
        known_nodes.append(row['node_id'])

known_nodes = sorted(list(set(known_nodes)))
unknown_nodes = sorted(list(set(unknown_nodes)))
print(f"Known nodes: {len(known_nodes)}, Unknown nodes: {len(unknown_nodes)}")

Labels: ['Belarusians', 'Northen Russians', 'Southern Russians', 'Ukranians']


Processing nodes: 230887it [00:08, 27609.75it/s]

Known nodes: 4633, Unknown nodes: 226254


In [4]:
train_nodes_orig, temp_nodes = train_test_split(
    known_nodes, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
val_nodes_orig, test_nodes_orig = train_test_split(
    temp_nodes, test_size=VAL_TEST_RATIO, random_state=RANDOM_SEED
)

print(f"Original train nodes: {len(train_nodes_orig)}")
print(f"Original val nodes: {len(val_nodes_orig)}")
print(f"Original test nodes: {len(test_nodes_orig)}")


Original train nodes: 2779
Original val nodes: 927
Original test nodes: 927


In [5]:
back_labels = {val: i for i, val in enumerate(labels)}
back_labels

{'Belarusians': 0,
 'Northen Russians': 1,
 'Southern Russians': 2,
 'Ukranians': 3}

In [11]:
val_len = len(val_nodes_orig) // 2
test_len = len(test_nodes_orig) // 2

In [12]:
nodes = np.zeros((max_node + 1 + val_len + test_len, len(labels)), dtype=np.float32)
edges = defaultdict(list)

for _, row in tqdm(data.iterrows(), total=len(data), desc="Building features and edges"):
    if row['label_id1'] != 'masked':
        nodes[row['node_id1'], back_labels[row['label_id1']]] = 1.0
    else:
        nodes[row['node_id1']] = np.ones(len(labels)) / len(labels)
    if row['label_id2'] != 'masked':
        nodes[row['node_id2'], back_labels[row['label_id2']]] = 1.0
    else:
        nodes[row['node_id2']] = np.ones(len(labels)) / len(labels)
    edges[row['node_id1']].append(row['node_id2'])
    edges[row['node_id2']].append(row['node_id1'])

Building features and edges: 100%|██████████| 6802907/6802907 [06:26<00:00, 17610.12it/s]


In [13]:
def create_synthetic_nodes(original_nodes, nodes_array, edges_dict, start_id):
    shuffled = original_nodes.copy()
    random.shuffle(shuffled)
    
    pairs = []
    for i in range(0, len(shuffled) - 1, 2):
        node1 = shuffled[i]
        node2 = shuffled[i + 1]
        coef = random.randint(1, 9) / 10
        pairs.append((node1, node2, coef))
    
    synthetic_ids = []
    
    for i, (node1, node2, coef) in enumerate(pairs):
        cur_id = start_id + i
        synthetic_ids.append(cur_id)

        nodes_array[cur_id] = coef * nodes_array[node1] + (1 - coef) * nodes_array[node2]

        edges1 = edges_dict[node1]
        edges2 = edges_dict[node2]

        mask_edges1 = np.random.rand(len(edges1)) < coef
        mask_edges2 = np.random.rand(len(edges2)) < (1 - coef)

        new_edges = np.concatenate([
            np.array(edges1, dtype=np.int32)[mask_edges1],
            np.array(edges2, dtype=np.int32)[mask_edges2]
        ]).tolist()
        new_edges = list(set(new_edges))

        edges_dict[cur_id] = new_edges
        for u in new_edges:
            edges_dict[u].append(cur_id)
    
    return synthetic_ids, pairs

In [14]:
# Create synthetic nodes for VAL
val_start_id = max_node + 1
val_nodes_synthetic, val_pairs = create_synthetic_nodes(
    val_nodes_orig, nodes, edges, val_start_id
)
print(f"Created {len(val_nodes_synthetic)} val synthetic nodes")

# Create synthetic nodes for TEST
test_start_id = val_start_id + len(val_nodes_synthetic)
test_nodes_synthetic, test_pairs = create_synthetic_nodes(
    test_nodes_orig, nodes, edges, test_start_id
)
print(f"Created {len(test_nodes_synthetic)} test synthetic nodes")

Created 463 val synthetic nodes
Created 463 test synthetic nodes


In [15]:
train_nodes = sorted(train_nodes_orig)
val_nodes = sorted(val_nodes_synthetic)
test_nodes = sorted(test_nodes_synthetic)

print(f"\nFinal splits:")
print(f"Train nodes (original): {len(train_nodes)}")
print(f"Val nodes (synthetic): {len(val_nodes)}")
print(f"Test nodes (synthetic): {len(test_nodes)}")


Final splits:
Train nodes (original): 2779
Val nodes (synthetic): 463
Test nodes (synthetic): 463


In [16]:
node_labels_masked = nodes.copy()
for n in test_nodes:
    node_labels_masked[n] = np.ones(len(labels)) / len(labels)

In [17]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(os.path.join(OUTPUT_DIR, 'train_nodes.npy'), np.array(train_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'val_nodes.npy'), np.array(val_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'test_nodes.npy'), np.array(test_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'unknown_nodes.npy'), np.array(unknown_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'node_labels.npy'), nodes)
np.save(os.path.join(OUTPUT_DIR, 'node_labels_masked.npy'), node_labels_masked)

print(f"Saved node arrays to {OUTPUT_DIR}")

Saved node arrays to splits_v2


In [18]:
pd_pairs = []
for u, vs in tqdm(edges.items(), total=len(edges), desc="Building edge DataFrame"):
    for v in vs:
        pd_pairs.append((u, v))
pd_pairs = pd.DataFrame(pd_pairs, columns=['node_id1', 'node_id2'])
pd_pairs.to_csv(os.path.join(OUTPUT_DIR, 'edges_data.csv'), index=False)
print(f"Saved {len(pd_pairs)} edges")

Building edge DataFrame: 100%|██████████| 231813/231813 [00:02<00:00, 77836.86it/s] 


Saved 16367992 edges


In [19]:
with open(os.path.join(OUTPUT_DIR, 'labels.txt'), 'w') as f:
    for label in labels:
        f.write(f"{label}\n")
print(f"Saved {len(labels)} label names")

Saved 4 label names


In [20]:
test_ground_truth = {
    'node_ids': np.array(test_nodes, dtype=np.int32),
    'true_labels': np.array([nodes[n] for n in test_nodes], dtype=np.float32),
    'label_names': labels
}
with open(os.path.join(OUTPUT_DIR, 'test_ground_truth.pkl'), 'wb') as f:
    pickle.dump(test_ground_truth, f)
print("Saved test ground truth file")

Saved test ground truth file


In [21]:
print("\n" + "="*50)
print("DATASET GENERATION COMPLETE")
print("="*50)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nNode counts:")
print(f"  - Train (original nodes): {len(train_nodes)}")
print(f"  - Val (synthetic nodes): {len(val_nodes)}")
print(f"  - Test (synthetic nodes): {len(test_nodes)}")
print(f"  - Unknown nodes: {len(unknown_nodes)}")
print(f"\nTotal nodes in feature matrix: {nodes.shape[0]}")
print(f"Total edges: {len(pd_pairs)}")
print(f"\nNote: Training will create synthetic nodes on-the-fly from train_nodes")


DATASET GENERATION COMPLETE

Output directory: splits_v2

Node counts:
  - Train (original nodes): 2779
  - Val (synthetic nodes): 463
  - Test (synthetic nodes): 463
  - Unknown nodes: 226254

Total nodes in feature matrix: 248755
Total edges: 16367992

Note: Training will create synthetic nodes on-the-fly from train_nodes
